# LLMs in Practice: Hands-On Workshop

**Duration:** 90-120 minutes  
**Topic:** Design effective prompts and run cost-aware LLM inference

---

## Learning Objectives

By the end of this notebook, you will:
1. Write effective zero-shot and few-shot prompts
2. Understand and configure inference parameters
3. Calculate and optimize LLM costs
4. Build a production-ready prompt engineering system

---

## Setup

**Required:** OpenAI API key (or use mock mode for learning)

**Installation:**

In [ ]:
# Install required packages
!pip install openai tiktoken python-dotenv -q

In [ ]:
# Imports
import openai
import tiktoken
import os
import json
from typing import Dict, List, Optional
from datetime import datetime

print("✓ Imports successful")

### API Key Configuration

**Option 1:** Use environment variable (recommended)
```python
# Set in terminal: export OPENAI_API_KEY='your-key-here'
api_key = os.getenv('OPENAI_API_KEY')
```

**Option 2:** Direct assignment (for learning only)
```python
api_key = "your-api-key-here"  # Not recommended for production
```

**Option 3:** Mock mode (no API key needed)

In [ ]:
# Configuration
USE_MOCK_MODE = True  # Set to False when using real API

if USE_MOCK_MODE:
    print("🔧 Running in MOCK MODE (no API calls)")
    print("   Set USE_MOCK_MODE = False to use real OpenAI API")
    client = None
else:
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise ValueError("Please set OPENAI_API_KEY environment variable")
    client = openai.OpenAI(api_key=api_key)
    print("✓ OpenAI client initialized")

---

## Part 1: Zero-Shot Prompting (20 minutes)

### Theory Recap

Zero-shot prompting = asking the LLM to perform a task **without** providing examples.

**4-Part Structure:**
1. **Role/Context** - Who is the AI?
2. **Task** - What should it do?
3. **Constraints** - How should it do it?
4. **Format** - What should the output look like?

### Exercise 1.1: Email Classification (Zero-Shot)

**Goal:** Classify customer emails into categories

In [ ]:
def classify_email_zero_shot(email_text: str) -> str:
    """
    Classify email using zero-shot prompting
    """
    # TODO: Write an effective zero-shot prompt
    prompt = f"""
You are an email classification system for customer support.

Task: Classify the following email into ONE category.

Categories:
- Technical Support
- Billing Question
- Feature Request
- Bug Report
- General Inquiry

Constraints:
- Return ONLY the category name
- No explanations or extra text

Email:
{email_text}

Category:
"""

    if USE_MOCK_MODE:
        # Mock response
        mock_responses = {
            "password": "Technical Support",
            "charge": "Billing Question",
            "feature": "Feature Request",
            "crash": "Bug Report"
        }
        for keyword, category in mock_responses.items():
            if keyword in email_text.lower():
                return category
        return "General Inquiry"
    else:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=10
        )
        return response.choices[0].message.content.strip()

# Test cases
test_emails = [
    "I can't reset my password. The link doesn't work.",
    "Why was I charged twice this month?",
    "Can you add dark mode to the app?",
    "The app crashes when I upload photos."
]

print("Email Classification Results:")
print("=" * 60)
for email in test_emails:
    category = classify_email_zero_shot(email)
    print(f"Email: {email[:50]}...")
    print(f"Category: {category}")
    print("-" * 60)

### Exercise 1.2: Structured Data Extraction

**Goal:** Extract structured data from invoice text

In [ ]:
def extract_invoice_data(invoice_text: str) -> Dict:
    """
    Extract structured data from invoice using zero-shot prompting
    """
    prompt = f"""
You are a professional data extraction system for accounting software.

Task: Extract the following fields from the invoice:
- invoice_number
- invoice_date (format: YYYY-MM-DD)
- vendor_name
- total_amount (numeric only, no symbols)
- line_items (array of {{"description": str, "amount": float}})

Constraints:
- If field not found, use null
- Remove currency symbols from amounts
- Convert dates to YYYY-MM-DD format

Format: Return valid JSON only, no explanations.

Invoice:
{invoice_text}

JSON Output:
"""

    if USE_MOCK_MODE:
        # Mock extraction
        return {
            "invoice_number": "INV-2024-001",
            "invoice_date": "2024-03-15",
            "vendor_name": "ACME Corporation",
            "total_amount": 3050.00,
            "line_items": [
                {"description": "Web design", "amount": 2500.00},
                {"description": "Hosting", "amount": 500.00},
                {"description": "Domain", "amount": 50.00}
            ]
        }
    else:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=300
        )
        return json.loads(response.choices[0].message.content)

# Test invoice
sample_invoice = """
ACME Corporation
Invoice #: INV-2024-001
Date: March 15, 2024

Services rendered:
- Web design: $2,500
- Hosting (annual): $500
- Domain registration: $50

Total: $3,050
"""

result = extract_invoice_data(sample_invoice)
print("Extracted Invoice Data:")
print(json.dumps(result, indent=2))

### 💡 Practice Challenge 1

**Your Turn:** Write a zero-shot prompt to:
- Input: Product review text
- Output: Sentiment (Positive/Negative/Neutral) + Brief reason (5 words max)

In [ ]:
def analyze_sentiment_zero_shot(review: str) -> Dict:
    """
    TODO: Implement sentiment analysis with zero-shot prompting

    Return format:
    {
        "sentiment": "Positive" | "Negative" | "Neutral",
        "reason": "brief reason (5 words max)"
    }
    """
    # YOUR CODE HERE
    prompt = f"""
# TODO: Write your prompt here
"""

    # Mock response for testing
    if USE_MOCK_MODE:
        if "great" in review.lower() or "love" in review.lower():
            return {"sentiment": "Positive", "reason": "positive language used"}
        elif "bad" in review.lower() or "terrible" in review.lower():
            return {"sentiment": "Negative", "reason": "negative language used"}
        return {"sentiment": "Neutral", "reason": "balanced feedback"}

    # Real API call would go here
    pass

# Test your implementation
test_reviews = [
    "Great product! Exactly what I needed.",
    "Terrible quality. Broke after one day.",
    "It's okay, nothing special."
]

for review in test_reviews:
    result = analyze_sentiment_zero_shot(review)
    print(f"Review: {review}")
    print(f"Result: {result}")
    print("-" * 60)

---

## Part 2: Few-Shot Prompting (25 minutes)

### Theory Recap

Few-shot prompting = providing **examples** before the actual task.

**When to use:**
- Complex output formats
- Nuanced classifications
- Specific style/tone needed

### Exercise 2.1: Few-Shot Sentiment Analysis with Nuance

In [ ]:
def analyze_sentiment_few_shot(review: str) -> Dict:
    """
    Analyze sentiment using few-shot prompting
    """
    prompt = f"""
Classify customer feedback sentiment with nuance.

Example 1:
Review: "Amazing product! Exceeded expectations."
Output: {{"sentiment": "positive", "confidence": "high", "aspects": {{"product": "positive"}}}}

Example 2:
Review: "Product is okay, but customer service was terrible."
Output: {{"sentiment": "mixed", "confidence": "medium", "aspects": {{"product": "neutral", "service": "negative"}}}}

Example 3:
Review: "Returned it immediately. Complete waste of money."
Output: {{"sentiment": "negative", "confidence": "high", "aspects": {{"overall": "negative"}}}}

Now classify:
Review: "{review}"
Output:
"""

    if USE_MOCK_MODE:
        # Intelligent mock based on keywords
        review_lower = review.lower()

        if "amazing" in review_lower or "love" in review_lower:
            return {
                "sentiment": "positive",
                "confidence": "high",
                "aspects": {"product": "positive"}
            }
        elif "but" in review_lower:
            return {
                "sentiment": "mixed",
                "confidence": "medium",
                "aspects": {"product": "positive", "service": "negative"}
            }
        elif "terrible" in review_lower or "bad" in review_lower:
            return {
                "sentiment": "negative",
                "confidence": "high",
                "aspects": {"overall": "negative"}
            }
        return {
            "sentiment": "neutral",
            "confidence": "medium",
            "aspects": {"product": "neutral"}
        }
    else:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=150
        )
        return json.loads(response.choices[0].message.content)

# Test cases
test_reviews = [
    "Love the features but shipping took forever!",
    "Worst purchase ever. Don't waste your money.",
    "Good value for the price. Works as expected."
]

print("Few-Shot Sentiment Analysis Results:")
print("=" * 70)
for review in test_reviews:
    result = analyze_sentiment_few_shot(review)
    print(f"\nReview: {review}")
    print(f"Sentiment: {result['sentiment']}")
    print(f"Confidence: {result['confidence']}")
    print(f"Aspects: {result['aspects']}")

### Exercise 2.2: Format Consistency Test

**Goal:** Demonstrate importance of consistent example formatting

In [ ]:
# BAD: Inconsistent format
prompt_bad = """
Example 1:
Q: "Reset password"
A: Go to settings

Example 2:
Input: "Track order"
Output: {{"answer": "Check tracking page", "category": "shipping"}}

Example 3:
User → "Refund policy?"
Bot → 30 days
"""

# GOOD: Consistent format
prompt_good = """
Example 1:
Input: "Reset password"
Output: {{"answer": "Go to settings", "category": "account"}}

Example 2:
Input: "Track order"
Output: {{"answer": "Check tracking page", "category": "shipping"}}

Example 3:
Input: "Refund policy?"
Output: {{"answer": "30 days full refund", "category": "policy"}}
"""

print("❌ BAD Example (Inconsistent Format):")
print(prompt_bad)
print("\n" + "=" * 70 + "\n")
print("✓ GOOD Example (Consistent Format):")
print(prompt_good)

print("\n📌 Key Takeaway:")
print("Consistent format = Predictable, parseable outputs")
print("Inconsistent format = Unpredictable, hard to parse")

### 💡 Practice Challenge 2

**Your Turn:** Create a few-shot prompt for intent classification

**Categories:**
- order_status
- return_request
- product_inquiry
- complaint

Provide 3 examples (one per category), then classify new inputs.

In [ ]:
def classify_intent_few_shot(message: str) -> str:
    """
    TODO: Implement intent classification with few-shot prompting

    Categories: order_status, return_request, product_inquiry, complaint
    """
    # YOUR CODE HERE
    prompt = f"""
# TODO: Add 3 examples (one per category)
# Example format:
# Input: "..."
# Intent: "..."

Now classify:
Input: "{message}"
Intent:
"""

    # Mock implementation
    if USE_MOCK_MODE:
        message_lower = message.lower()
        if "where" in message_lower and "order" in message_lower:
            return "order_status"
        elif "return" in message_lower:
            return "return_request"
        elif "does" in message_lower or "how" in message_lower:
            return "product_inquiry"
        return "complaint"

    # Real implementation would go here
    pass

# Test your implementation
test_messages = [
    "Where is my order #12345?",
    "I want to return this item",
    "Does this come in blue?",
    "This is unacceptable! Poor quality!"
]

print("Intent Classification Results:")
print("=" * 60)
for msg in test_messages:
    intent = classify_intent_few_shot(msg)
    print(f"Message: {msg}")
    print(f"Intent: {intent}")
    print("-" * 60)

---

## Part 3: System Prompts & Parameters (20 minutes)

### Exercise 3.1: System Prompt Design

In [ ]:
class CustomerSupportBot:
    """
    Chatbot with system prompt for customer support
    """
    def __init__(self, use_mock=True):
        self.use_mock = use_mock
        self.system_prompt = """
You are SupportBot, a friendly customer service agent for TechStore.

CAPABILITIES:
- Answer product questions
- Check order status
- Process returns (within 30 days)

CONSTRAINTS:
- Cannot access payment information
- Cannot make exceptions to 30-day return policy
- Escalate to human if customer is angry

TONE: Friendly, professional, empathetic

RESPONSE STRUCTURE:
1. Greet and acknowledge
2. Provide answer/solution
3. Ask if they need more help
"""
        self.conversation_history = []

    def chat(self, user_message: str) -> str:
        """
        Process user message with system prompt
        """
        if self.use_mock:
            # Mock responses
            responses = {
                "password": "Hi! I'd be happy to help you reset your password. Please visit www.techstore.com/reset and follow the instructions. Is there anything else I can help with?",
                "return": "Hello! I can help you with a return. Our policy allows returns within 30 days of purchase. Please provide your order number to proceed. Can I help with anything else?",
                "order": "Hi there! I can check your order status. Could you please provide your order number? Is there anything else you need?"
            }

            for keyword, response in responses.items():
                if keyword in user_message.lower():
                    return response

            return "Hello! I'm here to help. Could you provide more details about your question? Is there anything specific I can assist with?"
        else:
            # Real API call
            self.conversation_history.append(
                {"role": "user", "content": user_message}
            )

            messages = [
                {"role": "system", "content": self.system_prompt}
            ] + self.conversation_history

            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages,
                temperature=0.7,
                max_tokens=150
            )

            assistant_message = response.choices[0].message.content
            self.conversation_history.append(
                {"role": "assistant", "content": assistant_message}
            )

            return assistant_message

# Test the bot
bot = CustomerSupportBot(use_mock=USE_MOCK_MODE)

test_queries = [
    "I can't reset my password",
    "I want to return an item",
    "Where is my order?"
]

print("Customer Support Bot Demo:")
print("=" * 70)
for query in test_queries:
    print(f"\nUser: {query}")
    response = bot.chat(query)
    print(f"Bot: {response}")

### Exercise 3.2: Temperature Experimentation

In [ ]:
def test_temperature_impact(prompt: str, temperatures: List[float]):
    """
    Test how temperature affects output
    """
    print(f"Prompt: {prompt}")
    print("=" * 70)

    for temp in temperatures:
        print(f"\nTemperature: {temp}")
        print("-" * 70)

        if USE_MOCK_MODE:
            # Simulate temperature effects
            if temp < 0.3:
                output = "The capital of France is Paris."
            elif temp < 0.7:
                output = "The capital of France is Paris, a beautiful city."
            else:
                output = "The capital of France is Paris, the magnificent City of Light!"
            print(f"Output: {output}")
        else:
            # Run 3 times to see variation
            for i in range(3):
                response = client.chat.completions.create(
                    model="gpt-3.5-turbo",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temp,
                    max_tokens=50
                )
                print(f"Run {i+1}: {response.choices[0].message.content}")

# Test with different temperatures
test_prompt = "Complete this sentence: The capital of France is"
test_temps = [0.0, 0.5, 1.0]

test_temperature_impact(test_prompt, test_temps)

### Parameter Comparison Table

| Temperature | Top-P | Use Case | Example |
|-------------|-------|----------|----------|
| 0.0 | 0.1 | Code generation | Consistent syntax |
| 0.3 | 0.5 | Technical docs | Clear, accurate |
| 0.7 | 0.9 | Q&A | Natural responses |
| 1.0 | 0.95 | Creative writing | Varied, interesting |

---

## Part 4: Cost Analysis & Optimization (20 minutes)

### Exercise 4.1: Token Counting

In [ ]:
def count_tokens(text: str, model: str = "gpt-3.5-turbo") -> int:
    """
    Count tokens in text using tiktoken
    """
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")

    return len(encoding.encode(text))

# Test token counting
test_texts = [
    "Hello, world!",
    "I'll use GPT-4 for my AI chatbot",
    "The quick brown fox jumps over the lazy dog"
]

print("Token Counting Examples:")
print("=" * 70)
for text in test_texts:
    tokens = count_tokens(text)
    words = len(text.split())
    ratio = tokens / words if words > 0 else 0

    print(f"\nText: {text}")
    print(f"Words: {words}")
    print(f"Tokens: {tokens}")
    print(f"Ratio: {ratio:.2f} tokens/word")

### Exercise 4.2: Cost Calculator

In [ ]:
class CostCalculator:
    """
    Calculate LLM API costs
    """
    PRICING = {
        "gpt-4": {
            "input": 0.03,   # per 1K tokens
            "output": 0.06
        },
        "gpt-3.5-turbo": {
            "input": 0.001,
            "output": 0.002
        }
    }

    def calculate_cost(
        self,
        input_tokens: int,
        output_tokens: int,
        model: str = "gpt-3.5-turbo"
    ) -> Dict:
        """
        Calculate cost for single request
        """
        pricing = self.PRICING[model]

        input_cost = (input_tokens / 1000) * pricing["input"]
        output_cost = (output_tokens / 1000) * pricing["output"]
        total_cost = input_cost + output_cost

        return {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "input_cost": input_cost,
            "output_cost": output_cost,
            "total_cost": total_cost,
            "model": model
        }

    def project_costs(
        self,
        avg_input_tokens: int,
        avg_output_tokens: int,
        requests_per_day: int,
        model: str = "gpt-3.5-turbo"
    ) -> Dict:
        """
        Project daily and monthly costs
        """
        single_request = self.calculate_cost(
            avg_input_tokens,
            avg_output_tokens,
            model
        )

        daily_cost = single_request["total_cost"] * requests_per_day
        monthly_cost = daily_cost * 30

        return {
            "cost_per_request": single_request["total_cost"],
            "daily_cost": daily_cost,
            "monthly_cost": monthly_cost,
            "requests_per_day": requests_per_day,
            "model": model
        }

# Example: Customer support chatbot
calc = CostCalculator()

scenario = {
    "avg_input_tokens": 350,  # System prompt + user message
    "avg_output_tokens": 150,
    "requests_per_day": 5000
}

print("Cost Analysis: Customer Support Chatbot")
print("=" * 70)
print(f"\nScenario:")
print(f"- Average input tokens: {scenario['avg_input_tokens']}")
print(f"- Average output tokens: {scenario['avg_output_tokens']}")
print(f"- Requests per day: {scenario['requests_per_day']:,}")

# Compare GPT-3.5 vs GPT-4
for model in ["gpt-3.5-turbo", "gpt-4"]:
    print(f"\n{model.upper()}:")
    print("-" * 70)

    projection = calc.project_costs(
        scenario["avg_input_tokens"],
        scenario["avg_output_tokens"],
        scenario["requests_per_day"],
        model
    )

    print(f"Cost per request: ${projection['cost_per_request']:.6f}")
    print(f"Daily cost: ${projection['daily_cost']:.2f}")
    print(f"Monthly cost: ${projection['monthly_cost']:.2f}")

# Calculate savings
gpt35_monthly = calc.project_costs(
    scenario["avg_input_tokens"],
    scenario["avg_output_tokens"],
    scenario["requests_per_day"],
    "gpt-3.5-turbo"
)["monthly_cost"]

gpt4_monthly = calc.project_costs(
    scenario["avg_input_tokens"],
    scenario["avg_output_tokens"],
    scenario["requests_per_day"],
    "gpt-4"
)["monthly_cost"]

savings = gpt4_monthly - gpt35_monthly
savings_pct = (savings / gpt4_monthly) * 100

print(f"\n💰 SAVINGS by using GPT-3.5-turbo:")
print(f"${savings:.2f}/month ({savings_pct:.1f}% reduction)")

### Exercise 4.3: Prompt Optimization

In [ ]:
# Compare verbose vs optimized prompts
prompts = {
    "verbose": """
I would greatly appreciate it if you could please take the time to
carefully and thoroughly analyze the following customer feedback and
provide me with a detailed summary of the main points that were mentioned,
including both positive and negative aspects, as well as any suggestions
for improvement that might have been indicated by the customer in their
feedback message.
""",
    "optimized": """
Analyze this customer feedback. Summarize:
1. Positive points
2. Negative points
3. Improvement suggestions
"""
}

print("Prompt Optimization Analysis:")
print("=" * 70)

for name, prompt in prompts.items():
    tokens = count_tokens(prompt)
    words = len(prompt.split())

    print(f"\n{name.upper()} Prompt:")
    print(f"Words: {words}")
    print(f"Tokens: {tokens}")

    # Cost at 10K requests/day
    daily_requests = 10000
    cost_gpt4 = (tokens / 1000) * 0.03 * daily_requests
    monthly_cost = cost_gpt4 * 30

    print(f"Monthly cost (GPT-4, 10K req/day): ${monthly_cost:.2f}")

# Calculate savings
verbose_tokens = count_tokens(prompts["verbose"])
optimized_tokens = count_tokens(prompts["optimized"])
token_reduction = verbose_tokens - optimized_tokens
reduction_pct = (token_reduction / verbose_tokens) * 100

monthly_savings = ((token_reduction / 1000) * 0.03 * 10000 * 30)

print(f"\n💡 OPTIMIZATION RESULTS:")
print(f"Token reduction: {token_reduction} ({reduction_pct:.1f}%)")
print(f"Monthly savings (10K req/day): ${monthly_savings:.2f}")

---

## Part 5: Final Project (30 minutes)

### Build a Production-Ready Review Analyzer

**Requirements:**
1. Classify sentiment (positive/negative/mixed/neutral)
2. Extract key issues
3. Suggest response
4. Assess urgency
5. Track costs
6. Stay within budget

In [ ]:
class ProductReviewAnalyzer:
    """
    Complete production-ready review analyzer
    """
    def __init__(self, use_mock=True, budget_limit=100.0):
        self.use_mock = use_mock
        self.budget_limit = budget_limit
        self.cost_tracker = CostCalculator()
        self.total_cost = 0.0
        self.reviews_processed = 0

        # System prompt + few-shot examples
        self.system_prompt = """
You are a product review analyzer. Return JSON only.

Example 1:
Review: "Love it! Best purchase ever."
Output: {
  "sentiment": "positive",
  "rating_estimate": 5,
  "key_issues": [],
  "response_suggestion": "Thank you for the wonderful feedback!",
  "urgency": "low"
}

Example 2:
Review: "Good quality but shipping was slow."
Output: {
  "sentiment": "mixed",
  "rating_estimate": 4,
  "key_issues": ["slow_shipping"],
  "response_suggestion": "Thanks for feedback. We're improving delivery times.",
  "urgency": "medium"
}
"""

    def analyze(self, review_text: str) -> Dict:
        """
        Analyze single review
        """
        # Check budget
        if self.total_cost >= self.budget_limit:
            return {
                "error": "Budget limit reached",
                "total_cost": self.total_cost,
                "budget_limit": self.budget_limit
            }

        user_message = f"Now analyze:\nReview: {review_text}"

        if self.use_mock:
            # Mock analysis
            result = self._mock_analyze(review_text)

            # Simulate token usage
            input_tokens = count_tokens(self.system_prompt + user_message)
            output_tokens = count_tokens(json.dumps(result))
        else:
            # Real API call
            messages = [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_message}
            ]

            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages,
                temperature=0.2,
                max_tokens=150
            )

            result = json.loads(response.choices[0].message.content)
            input_tokens = response.usage.prompt_tokens
            output_tokens = response.usage.completion_tokens

        # Track costs
        cost_info = self.cost_tracker.calculate_cost(
            input_tokens,
            output_tokens,
            "gpt-3.5-turbo"
        )

        self.total_cost += cost_info["total_cost"]
        self.reviews_processed += 1

        return {
            "analysis": result,
            "cost": cost_info,
            "total_cost_so_far": self.total_cost,
            "reviews_processed": self.reviews_processed
        }

    def _mock_analyze(self, review_text: str) -> Dict:
        """
        Mock analysis for testing without API
        """
        review_lower = review_text.lower()

        # Simple sentiment detection
        if any(word in review_lower for word in ["love", "great", "amazing", "excellent"]):
            sentiment = "positive"
            rating = 5
            urgency = "low"
            issues = []
            response = "Thank you for the wonderful feedback!"
        elif any(word in review_lower for word in ["terrible", "awful", "worst", "broke"]):
            sentiment = "negative"
            rating = 1
            urgency = "high"
            issues = ["product_quality"]
            response = "We're very sorry. Please email support@company.com for a refund."
        elif "but" in review_lower:
            sentiment = "mixed"
            rating = 3
            urgency = "medium"
            issues = ["shipping"] if "shipping" in review_lower else ["other"]
            response = "Thanks for feedback. We're working on improvements."
        else:
            sentiment = "neutral"
            rating = 3
            urgency = "low"
            issues = []
            response = "Thank you for your feedback."

        return {
            "sentiment": sentiment,
            "rating_estimate": rating,
            "key_issues": issues,
            "response_suggestion": response,
            "urgency": urgency
        }

    def get_report(self) -> Dict:
        """
        Generate cost and performance report
        """
        if self.reviews_processed == 0:
            return {"error": "No reviews processed"}

        avg_cost_per_review = self.total_cost / self.reviews_processed

        # Project to 3,000 reviews/day
        daily_projection = avg_cost_per_review * 3000
        monthly_projection = daily_projection * 30

        return {
            "reviews_processed": self.reviews_processed,
            "total_cost": f"${self.total_cost:.4f}",
            "avg_cost_per_review": f"${avg_cost_per_review:.6f}",
            "projected_daily_cost": f"${daily_projection:.2f}",
            "projected_monthly_cost": f"${monthly_projection:.2f}",
            "budget_limit": f"${self.budget_limit:.2f}",
            "within_budget": monthly_projection <= self.budget_limit,
            "budget_utilization": f"{(monthly_projection/self.budget_limit)*100:.1f}%"
        }

# Test the analyzer
analyzer = ProductReviewAnalyzer(use_mock=USE_MOCK_MODE, budget_limit=100.0)

test_reviews = [
    "Amazing product! Exactly what I needed.",
    "Terrible quality. Broke after 2 days.",
    "Good product but shipping took forever.",
    "It's okay, nothing special.",
    "Love the features but a bit expensive."
]

print("Product Review Analyzer Demo")
print("=" * 70)

for review in test_reviews:
    result = analyzer.analyze(review)

    if "error" in result:
        print(f"\nERROR: {result['error']}")
        break

    print(f"\nReview: {review}")
    print(f"Sentiment: {result['analysis']['sentiment']}")
    print(f"Issues: {result['analysis']['key_issues']}")
    print(f"Urgency: {result['analysis']['urgency']}")
    print(f"Response: {result['analysis']['response_suggestion']}")
    print(f"Cost: ${result['cost']['total_cost']:.6f}")

# Generate report
print("\n" + "=" * 70)
print("COST REPORT")
print("=" * 70)
report = analyzer.get_report()
for key, value in report.items():
    print(f"{key}: {value}")

### 🎯 Final Challenge

**Your Mission:** Optimize the analyzer to:
1. Reduce monthly cost to under $60
2. Maintain analysis quality

**Hints:**
- Reduce few-shot examples?
- Compress system prompt?
- Lower max_tokens?
- Use different model for some cases?

In [ ]:
# YOUR OPTIMIZATION CODE HERE

class OptimizedReviewAnalyzer(ProductReviewAnalyzer):
    """
    TODO: Optimize to reduce costs while maintaining quality

    Ideas to try:
    1. Shorter system prompt
    2. Fewer examples (2 instead of 3)
    3. Lower max_tokens (100 instead of 150)
    4. More concise examples
    """
    def __init__(self, use_mock=True, budget_limit=60.0):
        super().__init__(use_mock, budget_limit)

        # TODO: Write optimized system prompt here
        self.system_prompt = """
# YOUR OPTIMIZED PROMPT HERE
"""

# Test your optimization
# optimized_analyzer = OptimizedReviewAnalyzer(use_mock=USE_MOCK_MODE)
# ... run tests and check costs

---

## Summary & Key Takeaways

### ✅ What You Learned

**1. Zero-Shot Prompting:**
- 4-part structure: Role, Task, Constraints, Format
- Be specific and concise
- Define output format clearly

**2. Few-Shot Prompting:**
- Use for complex formats and nuanced tasks
- Keep examples consistent in format
- Include diverse scenarios
- Consider token cost of examples

**3. System Prompts:**
- Separate behavior rules from specific tasks
- Define capabilities and limitations
- Set tone and style guidelines

**4. Parameters:**
- Temperature: 0.0 (deterministic) to 1.0+ (creative)
- Top-p: Controls vocabulary diversity
- Max tokens: Cost control and output length

**5. Cost Optimization:**
- Token counting is essential
- Choose appropriate model (GPT-3.5 vs GPT-4)
- Compress prompts without losing clarity
- Set max_tokens limits
- Monitor and project costs

### 📊 Cost Optimization Checklist

- [ ] Count tokens before deploying
- [ ] Use GPT-3.5 for simple tasks
- [ ] Compress verbose prompts
- [ ] Set appropriate max_tokens
- [ ] Use caching when available
- [ ] Monitor costs in production
- [ ] Test different models for your use case

### 🚀 Next Steps

1. **Practice:** Build 3-5 different prompts for real use cases
2. **Experiment:** Test temperature ranges for your tasks
3. **Optimize:** Find the balance between quality and cost
4. **Deploy:** Start small, monitor, scale gradually

### 📚 Additional Resources

- OpenAI Cookbook: https://github.com/openai/openai-cookbook
- Prompt Engineering Guide: https://www.promptingguide.ai/
- Token Counter: https://platform.openai.com/tokenizer

---

**End of Workshop** 🎉